# Round 1 — carry-dominant re-render + the init A/B

Trains on **`strips_v3`** (38,091 strips, 73.3% carry) plus all three promoted **real** pools.
What changed vs `strips_v2_2` (full rationale: `docs/RUNG3.md` Step 4.1):

- **Carry (`measure`) mode replaces keysig and dominates**, rendered at written pitch (t0) wearing each makam's **conventional PRINTED signature** (`data/makam_signatures.json`, built from the adjudication-confirmed real labels). `every` mode is the minority and carries the transpose augmentation.
- **Slur distractors** — label-free arcs (≥3 notes, no "3") so the model stops reading every arc as `\tup3` (baseline tup3 precision **15%**, arc-triggered false-tup3 **77.6%**).
- **`--every-share`** re-weights every-vs-carry *within* the synthetic pool at train time. `every` is 26.7% of strips but supplies ~81% of all inline accidentals (4.4× the real rate) — a suspected driver of the komaSharp/komaFlat hallucination.

## Pre-registered rules (Step 4.0 — do not renegotiate mid-run)

- **Selection = ONE number:** free-running **real-val mean AEU F1**. Tie-break: arc-triggered false-`\tup3` rate. NOT teacher-forced val loss.
- **Sequential budget, 4 runs:** init A/B at fixed **s=0.15** first → then sweep **s ∈ {0.267, 0.05}** on the winning recipe only.
- **Exam ONCE**, on the winner, at the very end. Exam strips are deliberately NOT in this kit.


In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy the data package Drive -> VM disk and unzip (fast local disk for the dataloader).
%%time
!cp /content/drive/MyDrive/tnc/tnc_round1_colab.zip /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/tnc_round1_colab.zip
!ls /content/tnc/src/vision/ | head
!wc -l /content/tnc/data/synthetic/strips_v3/manifest.jsonl

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# SHAKEOUT (~3 min): 150 tiny steps from BASE — a WIRING smoke, not a result.
# Expect: `vocab: +25 tokens -> 100 ids`, the three real pools listed, the line
#   `every-share -> 0.150 of synthetic (was 0.269)`, and val loss FALLING.
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v3 --split data/split_v3.json --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/round1-shakeout \
    --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
    --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2

In [ ]:
# ===== CALIBRATE THROUGHPUT ON *THIS* RUNTIME (~2-3 min) — do this before any long run =====
# Measured on T4/2-vCPU/num-workers 2: ~2.4 s/step @ batch 8 = ~3.3 samples/s -> ~9 h for Arm B.
# That was augmentation-CPU-bound, not GPU-bound. On L4 (8 vCPU) with --num-workers 10 it should be
# several times faster. Read the s/step below and compute your OWN budget instead of trusting an
# estimate:   hours = (steps * batch) / samples_per_sec / 3600
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc   # vCPU count -> the ceiling on --num-workers
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v3 --split data/split_v3.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 \
    --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/calib \
    --lr 3e-5 --warmup-steps 20 --max-steps 60 --batch-size 16 \
    --limit-val 8 --eval-every 60 --save-every 60 --log-every 20 --num-workers 10

# OPTIONAL A/B on the bottleneck: rerun the same command with --no-augment.
# Much faster => CPU/augmentation-bound (raise --num-workers / bigger runtime).
# Barely faster => GPU-bound (only a bigger GPU or fewer steps helps).

In [ ]:
# ===== ARM B — single-stage JOINT from BASE (the control) =====
# ~35.4k train items; 7000 steps @ batch 16 ~= 3 epochs (the Rung-2 coverage convention).
#   TIME: derive from the calibration cell. Measured T4/2vCPU = 3.3 samples/s -> ~9 h (!).
#   L4 with --num-workers 10 should be several x faster; CONFIRM before committing.
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v3 --split data/split_v3.json --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/drive/MyDrive/tnc/round1-B-single \
    --lr 3e-5 --max-steps 7000 --batch-size 16 --num-workers 10  # set to ~nproc-2 (L4 High-RAM = 12 vCPU); T4 (2 vCPU) use 2

In [ ]:
# ===== ARM A, STAGE 1 — carry-dominant SYNTHETIC ONLY, from BASE =====
# No --real-dir: this builds the carry-native synthetic checkpoint that stage 2 specialises.
#   TIME: derive from the calibration cell (see Arm B note).
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v3 --split data/split_v3.json \
    --every-share 0.15 --out-dir /content/drive/MyDrive/tnc/round1-A-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10  # set to ~nproc-2 (L4 High-RAM = 12 vCPU); T4 (2 vCPU) use 2

In [ ]:
# ===== ARM A, STAGE 2 — real-SPECIALISATION fine-tune from stage 1 =====
# Round-0.5 recipe: fresh LOW lr + short warmup, starting from the stage-1 checkpoint.
# The :8 suffix oversamples each real pool so real is ~33% of batches. Without it real is
# only ~5.9%, meaning each real strip would be seen <1x in 2000 steps — which could never
# reproduce the Round-0.5 effect this arm exists to test. (Selection caveat: oversampled real
# can overfit fast, and `best` is picked on a synth-dominated val mix — so evaluate BOTH
# best and last on real-val at selection time.)
%cd /content/tnc
!python src/vision/train.py --model /content/drive/MyDrive/tnc/round1-A-stage1/best \
    --strips-dir data/synthetic/strips_v3 --split data/split_v3.json \
    --real-dir data/real/rung3/strips_nota:8 \
    --real-dir data/real/rung3/strips_r1:8 \
    --real-dir data/real/rung3/strips_tup:8 \
    --every-share 0.15 --out-dir /content/drive/MyDrive/tnc/round1-A-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 2000 --batch-size 16 --num-workers 10

In [ ]:
# RESUME after a disconnect: re-run cells 1-4, then this with the SAME flags as the arm you ran.
# --resume reloads model+optimizer+scheduler from <out-dir>/last and ignores --model.
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v3 --split data/split_v3.json --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/drive/MyDrive/tnc/round1-B-single \
    --lr 3e-5 --max-steps 7000 --batch-size 16 --num-workers 10  # set to ~nproc-2 (L4 High-RAM = 12 vCPU); T4 (2 vCPU) use 2 --resume

In [ ]:
# ===== SELECTION — the ONE pre-registered number =====
# Materialise the real-val pool (same stable-hash split train.py validated on), then read
# `MEAN F1` for each arm. Tie-break on the `ARC-\tup3` line. Do NOT look at the exam yet.
%cd /content/tnc
!python src/vision/make_realval_pool.py --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup --split data/split_v3.json

for arm in ["round1-A-stage2", "round1-B-single"]:
    print("=" * 70, "\n==", arm)
    !python src/vision/eval_omr.py --checkpoint /content/drive/MyDrive/tnc/{arm}/best \
        --strips-dir data/real/rung3/_realval --split none --show-errors 0

## After the init A/B

1. **Pick the winner on `MEAN F1`** (tie-break: the `ARC-\tup3` false-positive rate). Record both arms' numbers in `src/vision/MODEL_EVAL.md` — the loser's number is evidence, not waste.
2. **Then the every-share sweep** — rerun the WINNING recipe only, changing `--every-share` to `0.267` (as-rendered) and `0.05`, into separate `--out-dir`s. Select again on real-val mean AEU F1.
   - Diagnostic worth logging: per-class **precision** across the three s values. komaSharp's *share* actually rises as s falls, so if its precision doesn't move with s, the hallucination is **not** distributional and Round 2 needs a different lever.
   - Confound to state as a non-claim: s also scales transpose exposure, since `every` is the sole carrier of t≠0.
3. **Exam LAST, ONCE**, on the final winner — against the Step-4.0 floors (AEU ≥85%, mean-F1 ≥80%, per-class ≥75% recall AND ≥70% precision, tup3 precision ≥70%, SER ≤0.06, exact ≥45%, source gap ≤12pp, arc rate ≤10%). A criterion missed is written up as missed, never re-rolled on the same exam.
4. **Ship only on a clean pass:** download `best/` → ONNX export → int8 quantize → `onnx_parity.py` (fp32+int8) → `make_browser_gate.py` → browser gate → `apps/web/public/models/`.
